# Retail Sales - Exploratory Data Analysis

A fresher-level Data Science project: clean a messy retail sales export, explore it with Pandas, and summarize trends with basic statistics and visualizations.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter Notebook

**Dataset:** `data/raw/sales_raw.csv` -- synthetic Superstore-style sales data generated by `generate_sales_data.py`, with realistic data-quality issues (duplicates, missing values, inconsistent region casing) for the cleaning step to fix.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

## 2. Load the raw data

In [ ]:
df = pd.read_csv('data/raw/sales_raw.csv')
print(df.shape)
df.head()

In [ ]:
df.info()

## 3. Data quality check

Before cleaning, check for missing values and duplicate rows -- both are common in real exported data.

In [ ]:
print('Missing values per column:')
print(df.isna().sum())
print()
print('Duplicate rows:', df.duplicated().sum())

## 4. Data cleaning

- Drop exact duplicate rows
- Standardize inconsistent text casing in `region` (e.g. `WEST`, `west`, `West` should all be the same category)
- Convert numeric columns and coerce any bad values to `NaN`
- Fill missing `quantity` with the column median
- Fill missing `sales` with the **category median** (not the global median -- a Furniture sale and an Office Supplies sale are on different scales)
- Fill missing `profit` with the column median
- Fill missing `region` with `'Unknown'` rather than dropping the row

In [ ]:
df_clean = df.drop_duplicates().copy()

df_clean['region'] = df_clean['region'].str.title()

for col in ['quantity', 'sales', 'discount', 'profit']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

df_clean['quantity'] = df_clean['quantity'].fillna(df_clean['quantity'].median())
df_clean['sales'] = df_clean.groupby('category')['sales'].transform(
    lambda s: s.fillna(s.median())
)
df_clean['profit'] = df_clean['profit'].fillna(df_clean['profit'].median())
df_clean['region'] = df_clean['region'].fillna('Unknown')

print('Shape after cleaning:', df_clean.shape)
print('Remaining missing values:', df_clean.isna().sum().sum())

## 5. Descriptive statistics

In [ ]:
df_clean[['quantity', 'sales', 'discount', 'profit']].describe()

## 6. Sales by category

In [ ]:
sales_by_category = df_clean.groupby('category')['sales'].sum().sort_values(ascending=False)
print(sales_by_category)

plt.figure(figsize=(7, 4))
sns.barplot(x=sales_by_category.values, y=sales_by_category.index, hue=sales_by_category.index, palette='Blues_d', legend=False)
plt.title('Total Sales by Category')
plt.xlabel('Total Sales')
plt.tight_layout()
plt.savefig('images/sales_by_category.png', dpi=120)
plt.show()

## 7. Profit by region

In [ ]:
profit_by_region = df_clean.groupby('region')['profit'].sum().sort_values(ascending=False)
print(profit_by_region)

plt.figure(figsize=(7, 4))
sns.barplot(x=profit_by_region.index, y=profit_by_region.values, hue=profit_by_region.index, palette='Greens_d', legend=False)
plt.title('Total Profit by Region')
plt.ylabel('Total Profit')
plt.tight_layout()
plt.savefig('images/profit_by_region.png', dpi=120)
plt.show()

## 8. Sales share by customer segment

In [ ]:
sales_by_segment = df_clean.groupby('segment')['sales'].sum()

plt.figure(figsize=(5, 5))
plt.pie(sales_by_segment, labels=sales_by_segment.index, autopct='%1.1f%%', startangle=90)
plt.title('Sales Share by Customer Segment')
plt.tight_layout()
plt.savefig('images/sales_by_segment.png', dpi=120)
plt.show()

## 9. Monthly sales trend

In [ ]:
df_clean['month'] = df_clean['order_date'].str.slice(0, 7)
monthly_sales = df_clean.groupby('month')['sales'].sum().sort_index()

plt.figure(figsize=(9, 4))
monthly_sales.plot(marker='o')
plt.title('Monthly Sales Trend (2024)')
plt.ylabel('Total Sales')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('images/monthly_sales_trend.png', dpi=120)
plt.show()

## 10. Correlation between numeric features

In [ ]:
corr = df_clean[['quantity', 'sales', 'discount', 'profit']].corr()

plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig('images/correlation_heatmap.png', dpi=120)
plt.show()

## 11. Key insights

Based on a run of this notebook against the generated dataset (`generate_sales_data.py`, seed=7):

- **Furniture** had the highest total sales of the three categories, narrowly ahead of Technology, with Office Supplies lowest.
- **South** region generated the highest total profit; the small number of rows with an originally missing region (`Unknown`) contributed very little profit, confirming they don't skew the regional comparison.
- **Consumer** was the largest segment by total sales, followed closely by Home Office and Corporate.
- Sales and profit are **positively but moderately correlated** (~0.45) -- higher-value orders tend to be more profitable, but discounting clearly pulls some high-sales orders into a loss, which is visible in the correlation heatmap.
- Average discount across all orders is roughly **10-11%**.

*(Re-running the notebook will reproduce these exact numbers, since the raw data generation uses a fixed random seed.)*

## 12. Possible next steps

- Load `df_clean` into a MySQL table and re-run the same aggregations as SQL `GROUP BY` queries, to compare the two approaches.
- Add a simple linear regression (`scikit-learn`) predicting `profit` from `sales`, `discount`, and `quantity` as a basic ML workflow exercise.
- Export `df_clean` to CSV and build a Power BI dashboard on top of it.